# 14 · Ensembles — solutions (notebook exécuté)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/ch14_ensembles/05_solutions.ipynb)

Les réponses des exercices, exécutées. Les démarches détaillées (le *pourquoi*, les erreurs fréquentes, les variantes) sont dans `05_solutions.md`. Les cellules marquées `answer` enregistrent les réponses vérifiées par `wb.check` (`tools/build_answers.py`).

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | 14.Q1 à 14.Q11, 14.R1 à 14.R3, 14.1 à 14.3, 14.5 à 14.7 | vérifier tes réponses courtes | 🧠 🔁 ✏️ ∂ | ★ à ★★★ | |
| A | 14.11 à 14.14 | faire voter trois classifieurs du ch. 13, le vote et le bootstrap dans mylearn.ensemble, un jury de votants indépendants puis corrélés, voting et stacking (*prochaine session de génération*) | | | |
| B | 14.15 à 14.19 | forêts de scikit-learn sur California, nombre d'arbres face au bruit, BaggingClassifier, RandomForestClassifier et ExtraTreesClassifier dans mylearn.ensemble, importance des features (*prochaine session de génération*) | | | |
| C | 14.20 à 14.25 | courbes d'un boosting, learning rate et nombre d'arbres, frontières et débogage d'AdaBoost, AdaBoostClassifier et GradientBoostingRegressor dans mylearn.ensemble (*prochaine session de génération*) | | | |
| D | 14.26 à 14.28 | HistGradientBoosting, XGBoost et LightGBM sur California, un benchmark reproductible, le défi RMSE ≤ 0,45 (*prochaine session de génération*) | | | |

In [1]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("ref")  # reference implementation

🧪 Deep Learning Workbook : environnement
   Plateforme : Local (Linux x86_64, 2 CPU)
   Python 3.13.13 | numpy 2.1.3 | pandas 2.2.3 | matplotlib 3.10.0 | scikit-learn 1.6.1 | scipy 1.16.3 | torch 2.11.0+cpu | torchvision 0.26.0+cpu
   Device : cpu | Graine : 42 | FAST_MODE : True


## Objectifs et rappel express

- Calculer à la main un vote pondéré, la probabilité qu'une majorité se trompe, la part des exemples oubliés par un bootstrap, une itération d'AdaBoost et deux itérations de gradient boosting.
- Écrire et vérifier contre scikit-learn les ensembles de ta librairie, construits sur tes arbres du ch. 13 : `plurality_vote`, `bootstrap_indices`, `BaggingClassifier`, `RandomForestClassifier`, `ExtraTreesClassifier`, `AdaBoostClassifier` et `GradientBoostingRegressor`.
- Utiliser les forêts de scikit-learn, HistGradientBoosting, XGBoost et LightGBM avec early stopping, et interpréter l'importance des features.
- Choisir entre bagging et boosting, et rendre un benchmark reproductible.

**Rappel express.** Vote pondéré : $\hat y = \arg\max_k \sum_m w_m\,\mathbb{1}[h_m(\mathbf{x}) = k]$ ; majorité de $M$ votants indépendants : $\sum_{j > M/2} \binom{M}{j} p^j (1 - p)^{M - j}$ ; jamais tiré par un bootstrap : $(1 - 1/n)^n \to e^{-1} \approx 0{,}368$ ; variance d'une moyenne corrélée : $\rho\sigma^2 + (1 - \rho)\sigma^2/M$ ; AdaBoost (SAMME) : $\varepsilon_m = \sum_i w_i \mathbb{1}[h_m(\mathbf{x}_i) \ne y_i] / \sum_i w_i$, $\alpha_m = \eta\big(\ln\frac{1 - \varepsilon_m}{\varepsilon_m} + \ln(K - 1)\big)$, $w_i \leftarrow w_i e^{\alpha_m}$ pour les mal classés ; gradient boosting : $F_0 = \bar y$, $r_i = y_i - F_{m-1}(\mathbf{x}_i)$, $F_m = F_{m-1} + \eta\, h_m$. En scikit-learn : `sklearn.ensemble` (`BaggingClassifier`, `RandomForestClassifier`, `ExtraTreesClassifier`, `AdaBoostClassifier`, `GradientBoostingRegressor`, `HistGradientBoostingRegressor`, `VotingClassifier`, `StackingClassifier`), `sklearn.inspection.permutation_importance` ; `xgboost.XGBRegressor`, `lightgbm.LGBMRegressor`.

## Partie 0 · Vérifier tes réponses courtes (quiz, rappels, ✏️ 14.1, 14.2, 14.5 à 14.7 et ∂ 14.3)

Fais d'abord les quiz, les rappels et les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse courte ici : **la valeur** que tu as trouvée, pas l'expression Python, sinon tu ne vérifies rien. Un nombre : `42` ou `0.3` (en Python, le séparateur décimal est un **point** ; `0,3` sans guillemets serait un couple de deux nombres) ; un vrai ou faux : `True` ou `False` ; un choix : la lettre seule, entre guillemets (`"E"`) ; une suite de codes : les lettres dans l'ordre (`"XYZ"`) ; plusieurs nombres : une liste (`[4, 7]`). Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les questions « dans ta copie », la preuve ∂ 14.4, la réflexion (🗣️ 14.8, ⚖️ 14.9, 📄 14.10) et l'entretien se corrigent avec `05_solutions.md`.

In [2]:
# The short answers only use the numbers of their statements (02_exercices.md)
import math

import numpy as np
from scipy.stats import binom


def gini(weights_per_class):
    p = np.asarray(weights_per_class, dtype=float) / np.sum(weights_per_class)
    return float(1 - np.sum(p ** 2))


def majority_right(M, p):
    """P(more than M/2 of M independent voters are right), each right with probability p."""
    return float(binom.sf(M // 2, M, p))


# Q3: seven voters, their classes and weights
VOTES_Q3 = np.array(list("ABCABAC"))
W_Q3 = np.array([1, 2, 1, 1, 2, 1, -1])
SCORE_Q3 = {c: float(W_Q3[VOTES_Q3 == c].sum()) for c in "ABC"}

# 14.1: five voters (rows) and four samples (columns)
P_141 = np.array([[0, 1, 2, 0], [1, 1, 0, 2], [1, 0, 2, 2], [0, 1, 1, 0], [2, 0, 0, 1]])
W_141 = np.array([2, 1, 1.5, -1, 0.5])


def scores_141(weights):
    """(4 samples, 3 classes) table of weighted votes."""
    return np.array([[weights[P_141[:, j] == k].sum() for k in range(3)] for j in range(4)])


def winners_141(weights):
    return [int(k) for k in np.argmax(scores_141(weights), axis=1)]      # argmax: the smallest of tied classes


# 14.2: three weak learners A (x1 >= 1), B (x2 >= 1), C (x1 + x2 <= 3)
def score_142(x1, x2, wA=2, wB=2, wC=-3):
    return wA * (x1 >= 1) + wB * (x2 >= 1) + wC * (x1 + x2 <= 3)


REGIONS_142 = {"ABC": (1.5, 1.2), "AB": (2.5, 2.5), "AC": (2, 0.5), "A": (3, 0.5), "BC": (0.5, 1.5),
               "B": (0.5, 2.8), "C": (0.5, 0.5)}                      # one point inside each region
POINTS_142 = [(3, 0.5), (0.5, 2.8), (2, 0.5), (0.2, 1.5), (2.5, 2.5)]
TRUE_142 = [1, 1, 0, 0, 1]

# 14.6: AdaBoost by hand, eight points, stumps "x <= s -> g, else the other class"
X_146 = np.arange(1, 9)
Y_146 = np.array([0, 1, 0, 0, 1, 1, 1, 1])


def stump_146(s, g):
    return np.where(X_146 <= s, g, 1 - g)


def best_stump_146(w):
    candidates = [(float(w[stump_146(s, g) != Y_146].sum()), s, g) for s in np.arange(1.5, 8.5) for g in (0, 1)]
    return min(candidates)                                             # unique minimum here


def adaboost_146(rounds, stumps=None):
    """Errors, alphas, stumps and the weights before each round (learning_rate = 1, two classes)."""
    w = np.full(8, 1 / 8)
    out = []
    for m in range(rounds):
        if stumps and m < len(stumps):
            s, g = stumps[m]
            err = float(w[stump_146(s, g) != Y_146].sum())
        else:
            err, s, g = best_stump_146(w)
        alpha = math.log((1 - err) / err)
        out.append({"err": err, "alpha": alpha, "s": float(s), "g": int(g), "w": w.copy()})
        w = np.where(stump_146(s, g) != Y_146, w * math.exp(alpha), w)
        w = w / w.sum()
    return out, w


def vote_146(rounds):
    votes = np.zeros((8, 2))
    for r in rounds:
        votes[np.arange(8), stump_146(r["s"], r["g"])] += r["alpha"]
    return [int(k) for k in np.argmax(votes, axis=1)]


R_146, W_AFTER_1 = adaboost_146(1)
R2_146, _ = adaboost_146(2)
R3_146, _ = adaboost_146(3, stumps=[(R2_146[0]["s"], R2_146[0]["g"]), (R2_146[1]["s"], R2_146[1]["g"]), (2.5, 1)])

# 14.7: gradient boosting by hand, regression stumps, learning rate 0.5
X_147 = np.arange(1.0, 6.0)
Y_147 = np.array([1.0, 2.0, 6.0, 7.0, 9.0])


def stump_147(r):
    """Best regression stump on the residuals r: (threshold, left value, right value)."""
    best = None
    for s in (1.5, 2.5, 3.5, 4.5):
        left, right = r[X_147 <= s], r[X_147 > s]
        sse = ((left - left.mean()) ** 2).sum() + ((right - right.mean()) ** 2).sum()
        if best is None or sse < best[0]:
            best = (sse, s, float(left.mean()), float(right.mean()))
    return best[1:]


def boost_147(eta, stages):
    F = np.full(5, Y_147.mean())
    trees = []
    for _ in range(stages):
        s, vl, vr = stump_147(Y_147 - F)
        trees.append((s, vl, vr))
        F = F + eta * np.where(X_147 <= s, vl, vr)
    return F, trees


def predict_147(x, eta, trees):
    return float(Y_147.mean() + eta * sum(vl if x <= s else vr for s, vl, vr in trees))


F1_147, T1_147 = boost_147(0.5, 1)
F2_147, T2_147 = boost_147(0.5, 2)

**14.Q1 — Pourquoi un groupe peut battre chacun de ses membres**

In [3]:
wb.record("14.Q1a", majority_right(15, 0.7) > 0.7, mistakes={"relis l'encadré sur la loi binomiale : que devient la majorité de votants indépendants justes à plus de 50 % ?": False})
wb.record("14.Q1b", "B", mistakes={"des membres identiques font-ils jamais des erreurs différentes ?": "A",
          "la profondeur des arbres crée-t-elle des membres différents ?": "C",
          "qu'apporte un nombre pair de votants, à deux classes ?": "D"})
wb.record("14.Q1c", False, mistakes={"dix votes toujours identiques peuvent-ils être en désaccord avec l'arbre seul ?": True})
wb.record("14.Q1d", True, mistakes={"quand une copie tombe en panne, combien de copies justes restent, et que décide la majorité ?": False})

WB_ANSWER {"hash": "0989dcdb16daaf222c6652089b68ac6b3ee17fa50095e1b6b2c6204dfe5f1b57", "id": "14.Q1a", "kind": "bool", "mistakes": {"44e5ffde3085f5d43417b63d6d13f886b828e4178c9a98f93136a7bd541d4e40": "relis l'encadré sur la loi binomiale : que devient la majorité de votants indépendants justes à plus de 50 % ?"}}
📝 Ex 14.Q1a : réponse enregistrée (bool).
WB_ANSWER {"hash": "997d7250926088383fc980dc8145f73ed7ad04a72deab50142fd84a9ea450770", "id": "14.Q1b", "kind": "str", "mistakes": {"f1fad24c9accb0df93c308a2e604ff7092bd38f55b8bfff4902d5f5882c66c97": "des membres identiques font-ils jamais des erreurs différentes ?", "fc1cc5f0727692283d05243a7c468f19b6793636095551d00adbf114efb3fb14": "qu'apporte un nombre pair de votants, à deux classes ?", "ff722262afbbdf9a9c0430fbb3ace789da05c2c77d370d16f9dcdeabe98983cc": "la profondeur des arbres crée-t-elle des membres différents ?"}}
📝 Ex 14.Q1b : réponse enregistrée (str).
WB_ANSWER {"hash": "a612bc432de13c3e67ef79318a86fa688b19085901ffa41ff21a775

**14.Q2 — Un ensemble corrige-t-il un biais des données ?**

In [4]:
wb.record("14.Q2a", False, mistakes={"que deviennent les proportions de ce groupe dans un bootstrap des mêmes données ?": True})
wb.record("14.Q2b", "B", mistakes={"un bootstrap peut-il contenir un groupe absent des données de départ ?": "A",
          "entraîner plus longtemps change-t-il ce que contiennent les données ?": "C",
          "tirer des features change-t-il les exemples sur lesquels on apprend ?": "D"})
wb.record("14.Q2c", True, mistakes={"relis le paragraphe sur les ensembles hors du ML (§14.2) : que fait-on varier en prévision d'ensemble ?": False})
wb.record("14.Q2d", True, mistakes={"une erreur que tous les membres font varie-t-elle d'un membre à l'autre ?": False})

WB_ANSWER {"hash": "071febf324cd88b219bf7cc991133e7d13250694088e97d3897109f79297e2b7", "id": "14.Q2a", "kind": "bool", "mistakes": {"4bf6c46aaea2faac82601502f3fd675e34559e9c7107cb3f6392bb7292002ffe": "que deviennent les proportions de ce groupe dans un bootstrap des mêmes données ?"}}
📝 Ex 14.Q2a : réponse enregistrée (bool).
WB_ANSWER {"hash": "089dc494635dbdc1b5590c0e20c1d1d6e6d923dcf4a0e296e75a29219d05f76d", "id": "14.Q2b", "kind": "str", "mistakes": {"1342d2bca7afaa4f1542d398a62360c070445d8d6f7daf7faa7d3e088b4c17a1": "tirer des features change-t-il les exemples sur lesquels on apprend ?", "b9c18f5f3f4a3b51ab9074196320026fc3cc1fb7d5154144a22075060745b1bf": "entraîner plus longtemps change-t-il ce que contiennent les données ?", "fb350e850cc3f0d156fb60a6c86ef144a53ba3ef64b767da95dad74f9d1cf5b5": "un bootstrap peut-il contenir un groupe absent des données de départ ?"}}
📝 Ex 14.Q2b : réponse enregistrée (str).
WB_ANSWER {"hash": "d876f9419eeb4afd5a618f5d297dc65e240128b2d369f8f59c54153

**14.Q3 — Pluralité, majorité et vote pondéré**

In [5]:
wb.record("14.Q3a", "A", mistakes={"compte de nouveau les voix de chaque classe, sans les poids": "B",
          "compte les voix de chaque classe : C en a-t-elle le plus ?": "C"})
wb.record("14.Q3b", False, mistakes={"plus de la moitié de 7 voix, c'est au moins combien de voix ?": True})
wb.record("14.Q3c", int(SCORE_Q3['C']), mistakes={"le votant 6 a un poids négatif : as-tu retiré son poids ?": 2,
          "additionne les poids de tous les votants de C, signe compris": 1})
wb.record("14.Q3d", "B", mistakes={"avec les poids, recalcule le total de chaque classe": "A",
          "recalcule le total de chaque classe, poids et signes compris": "C"})

WB_ANSWER {"hash": "293680f478c7ae24649b91a13d345bf411827e6a4c4c5f30775bbf0952dc0b1a", "id": "14.Q3a", "kind": "str", "mistakes": {"7eb230fa5a1882b66eae0791e17160766d5cdfb48f4dc7ae3c0e675e049081ad": "compte les voix de chaque classe : C en a-t-elle le plus ?", "ac270beb34036edf2ce9260bfbe7968488533a0d9c45bea97df20011db9373fd": "compte de nouveau les voix de chaque classe, sans les poids"}}
📝 Ex 14.Q3a : réponse enregistrée (str).
WB_ANSWER {"hash": "7186bc912769cbbe3321cf9e91399e3c5e680d24206f0da2250b05570925788f", "id": "14.Q3b", "kind": "bool", "mistakes": {"cab951d41110cf307370a761090fee668681c9d3071de05c8521642b1f7bad6f": "plus de la moitié de 7 voix, c'est au moins combien de voix ?"}}
📝 Ex 14.Q3b : réponse enregistrée (bool).
WB_ANSWER {"hash": "0b0e1ffe68184dfb13f0119d8ab0b3cf393a50d217c62b2e16b49f62060254d0", "id": "14.Q3c", "kind": "int", "magnitude": null, "mistakes": {"1c5530a1782c60e5f5adc47b54e68143e206d6a6ccd21abd93b0b434b6caf45e": "additionne les poids de tous les votant

**14.Q4 — Tirer avec remise**

In [6]:
wb.record("14.Q4a", True, mistakes={"avec remise, un exemple tiré revient dans l'urne : peut-il ressortir ?": False})
wb.record("14.Q4b", 1, mistakes={"cherche le cas extrême : que se passe-t-il si les quatre tirages donnent le même exemple ?": 4,
          "les tirages identiques comptent pour un seul exemple distinct": 2})
wb.record("14.Q4c", 4, mistakes={"on demande le plus PETIT nombre d'exemples jamais tirés : combien quatre tirages peuvent-ils en toucher au mieux ?": 7,
          "8 exemples, au plus combien de tirés : combien en reste-t-il dehors ?": 0})
wb.record("14.Q4d", True, mistakes={"relis l'encadré sur la limite de (1 − 1/n)^n : quelle part des exemples n'est jamais tirée ?": False})
wb.record("14.Q4e", True, mistakes={"sans remise, un exemple peut-il sortir deux fois, ou rester dehors avec n tirages parmi n ?": False})

WB_ANSWER {"hash": "408fdb36264ef5b70c5279db1b2a8f4f5d43c55ec81b1eb92a8215e41301370e", "id": "14.Q4a", "kind": "bool", "mistakes": {"7e5ed9b81975fd7104377affd973bc312cb7ca86bb6174da444500814841df0c": "avec remise, un exemple tiré revient dans l'urne : peut-il ressortir ?"}}
📝 Ex 14.Q4a : réponse enregistrée (bool).
WB_ANSWER {"hash": "29ecb4234184574ef04deabbe50706441b5c333302c011ae004e1edc663ae5e3", "id": "14.Q4b", "kind": "int", "magnitude": 0, "mistakes": {"34c427e47d59971d2ce05f67b65c0b2f8c04eee32c85d60321332b30eba2dfc8": "cherche le cas extrême : que se passe-t-il si les quatre tirages donnent le même exemple ?", "e426bcc75d20ec44a35622aebcf367d830c9d851886205f1fce1a84faeceefd0": "les tirages identiques comptent pour un seul exemple distinct"}, "sign": 1}
📝 Ex 14.Q4b : réponse enregistrée (int).
WB_ANSWER {"hash": "ad712d534d8788f98d23512e94eb85857d35c0b7bdfd172fa24ea7f2b880b58c", "id": "14.Q4c", "kind": "int", "magnitude": 0, "mistakes": {"602aec9afb12e9b19e8a683124cda1c4d854bf0e

**14.Q5 — Ce que le bagging réduit**

In [7]:
wb.record("14.Q5a", "B", mistakes={"une moyenne de modèles biaisés de la même façon garde-t-elle ce biais ?": "A",
          "le bagging change-t-il les labels ?": "C",
          "prédire avec cent arbres est-il plus rapide qu'avec un seul ?": "D"})
wb.record("14.Q5b", "B", mistakes={"une régression linéaire change-t-elle beaucoup d'un bootstrap à l'autre ?": "A",
          "une souche change-t-elle beaucoup d'un bootstrap à l'autre ?": "C",
          "un kNN avec k = n prédit-il autre chose que la classe majoritaire ?": "D"})
wb.record("14.Q5c", False, mistakes={"un arbre voit-il les exemples que son bootstrap n'a pas tirés ?": True})
wb.record("14.Q5d", True, mistakes={"avec quels arbres prédit-on un exemple pour son score out-of-bag ?": False})

WB_ANSWER {"hash": "6627bcf35e3f9bec108d73df5b6fc905b169f85073fcb0b9ba5aaba563837573", "id": "14.Q5a", "kind": "str", "mistakes": {"1dcedb4bccc9050d77aa5c53437c1f9d8e6db4309a05a0884880866fc973fc19": "le bagging change-t-il les labels ?", "4bb95a5bfa1f2fbf82d3553b36e4ea4b413d1ca4e57654c62d49249cf7eb5199": "prédire avec cent arbres est-il plus rapide qu'avec un seul ?", "6ce106efa3e777194a75897f70fd97ef791f3a408bd53c3104c3c3b724cf9ade": "une moyenne de modèles biaisés de la même façon garde-t-elle ce biais ?"}}
📝 Ex 14.Q5a : réponse enregistrée (str).
WB_ANSWER {"hash": "d7da233329d03c39d37fc707c6bf9588ffb14bdd93df6b67bd17e56f97cbdb6f", "id": "14.Q5b", "kind": "str", "mistakes": {"b7d693001b8d1b681cb8fdb0336aca9784672a7bef8611bdce3670846d86e4e5": "une souche change-t-elle beaucoup d'un bootstrap à l'autre ?", "bb84747384ace90b1058d6a238a293544eb1524e0f17ba25fa4c4d67e611f94b": "une régression linéaire change-t-elle beaucoup d'un bootstrap à l'autre ?", "f37aef4bb3392a205a835d78a6a30d1e4e4

**14.Q6 — Le hasard d'une forêt aléatoire**

In [8]:
wb.record("14.Q6a", int(math.sqrt(16)), mistakes={"\"sqrt\" : la racine carrée du nombre de features, pas sa moitié": 8, "relis ce que veut dire \"sqrt\"": 16})
wb.record("14.Q6b", "C", mistakes={"relis le pseudo-code du §14.5 : à quelle ligne a lieu le tirage ?": "A",
          "un arbre tire-t-il ses features une seule fois ? Relis le pseudo-code": "B",
          "le tirage porte-t-il sur les exemples ou sur les features d'un nœud ?": "D"})
wb.record("14.Q6c", False, mistakes={"examiner deux fois la même feature à un nœud aurait-il un sens ?": True})
wb.record("14.Q6d", "AC", mistakes={"un seuil tiré au hasard : est-ce la forêt ou ExtraTrees ? Relis le §14.6": "BC",
          "l'ordre des exemples est-il une source de hasard voulue de la forêt ?": "CD",
          "relis le pseudo-code du §14.5 : que tire-t-on à chaque nœud ?": "AB"})
wb.record("14.Q6e", True, mistakes={"sans tirage de features, que reste-t-il d'aléatoire dans la forêt ?": False})

WB_ANSWER {"hash": "f14b12a3f9730b7a4899d2a50fcb4839a86d06d719ad1b34473ab1c5bfdfc263", "id": "14.Q6a", "kind": "int", "magnitude": 0, "mistakes": {"5b0a58eae7290a7ff1bd0af23ab37e662735bbfb749c75e4d29a2cb9d1856d4d": "\"sqrt\" : la racine carrée du nombre de features, pas sa moitié", "d0f5a37e242a4c1458479cf2d20b7ace0495a1aeb5641cb2474bc189417c6e9d": "relis ce que veut dire \"sqrt\""}, "sign": 1}
📝 Ex 14.Q6a : réponse enregistrée (int).
WB_ANSWER {"hash": "810a6f2d543bd75d6d2b401f80cf724930e4a9111936575aa3b276df29a12cc3", "id": "14.Q6b", "kind": "str", "mistakes": {"75114e05cf6ad018fb81719ff6717b346a0a6462822a3950e39d5b08f7a64f70": "relis le pseudo-code du §14.5 : à quelle ligne a lieu le tirage ?", "b6af852624deb495f7a924cb378590ebfd6d8dc0b8b54f13b343c158ab973b64": "le tirage porte-t-il sur les exemples ou sur les features d'un nœud ?", "c5575fb4f2a576cc70d702ff24b70820c9b1b2ed24420e66a5930cd17fefed14": "un arbre tire-t-il ses features une seule fois ? Relis le pseudo-code"}}
📝 Ex 14.Q6

**14.Q7 — ExtraTrees : encore plus de hasard**

In [9]:
wb.record("14.Q7a", "B", mistakes={"tester tous les seuils, c'est ce que fait une forêt aléatoire": "A",
          "la médiane dépend-elle du hasard ? Relis ce que tire ExtraTrees (§14.6)": "C",
          "les arbres d'ExtraTrees dépendent-ils les uns des autres ?": "D"})
wb.record("14.Q7b", True, mistakes={"relis la valeur par défaut de bootstrap pour ExtraTrees dans scikit-learn": False})
wb.record("14.Q7c", True, mistakes={"au même nœud, un seuil tiré peut-il faire baisser l'impureté plus que le meilleur seuil ?": False})
wb.record("14.Q7d", "B", mistakes={"ExtraTrees examine-t-il moins de features qu'une forêt, à max_features égal ?": "A",
          "un nœud d'ExtraTrees a-t-il des enfants ?": "C",
          "comment choisirait-il sa feature sans calculer d'impureté ?": "D"})

WB_ANSWER {"hash": "5565e85df6562d17244754fc72a09c91e4e92c39fbe20d7d50c3d9b73c095bc9", "id": "14.Q7a", "kind": "str", "mistakes": {"2ba4b72a3c77a2d9d16641839a8a24c6ff87ade0dc9743db60dd44380907427f": "la médiane dépend-elle du hasard ? Relis ce que tire ExtraTrees (§14.6)", "9547a5a517132fb2c1c71210beae48ab60748172ecfa7fd1bb57ca919c042bcb": "tester tous les seuils, c'est ce que fait une forêt aléatoire", "cfe0caf0ced013b8121ee8292980742aecac4039636d606a7ab9ed42b8157460": "les arbres d'ExtraTrees dépendent-ils les uns des autres ?"}}
📝 Ex 14.Q7a : réponse enregistrée (str).
WB_ANSWER {"hash": "f03d71aba7cb1935a707d0c14a7f5fc9fa24aab0423a81ce2ad595f7d1b01cf5", "id": "14.Q7b", "kind": "bool", "mistakes": {"d8cd01128f2be87f22e65b7b180f570162143fa0540058cddc7d40645aab15f1": "relis la valeur par défaut de bootstrap pour ExtraTrees dans scikit-learn"}}
📝 Ex 14.Q7b : réponse enregistrée (bool).
WB_ANSWER {"hash": "b2289fd6ac133f2341c2094d4e6a6a675099c966cd22be86f6f8dc198d67084a", "id": "14.Q7c"

**14.Q8 — Apprenant faible, souche et apprenant fort**

In [10]:
wb.record("14.Q8a", 2, mistakes={"une souche pose UNE question : combien de feuilles ?": 1, "une seule question, pas deux": 4})
wb.record("14.Q8b", 1, mistakes={"combien d'arêtes entre la racine et une feuille d'une souche ?": 0, "une souche a une seule question": 2})
wb.record("14.Q8c", True, mistakes={"à deux classes, que devient chaque erreur quand on échange les deux réponses ?": False})
wb.record("14.Q8d", "B", mistakes={"90 %, c'est un apprenant fort": "A",
          "exactement 50 %, c'est le hasard : est-ce « mieux que le hasard » ?": "C",
          "en dessous de 50 %, à deux classes, l'apprenant est pire que le hasard": "D"})
wb.record("14.Q8e", False, mistakes={"relis la dernière phrase du paragraphe sur les apprenants faibles et forts (§14.7)": True})

WB_ANSWER {"hash": "05beef76d19af2516747f5e0e9da18d50d3e212627a8702f9b85d8838a343b78", "id": "14.Q8a", "kind": "int", "magnitude": 0, "mistakes": {"12865ef878bde96e505e300ff484da934321972e0cb88b2f6cad8cc4f2fff1c0": "une souche pose UNE question : combien de feuilles ?", "b258d3b02170fca40a2b47966237161ba1f6d188d5816848a4b8be7058d2bdb3": "une seule question, pas deux"}, "sign": 1}
📝 Ex 14.Q8a : réponse enregistrée (int).
WB_ANSWER {"hash": "80f4800358fd7ef46e121b1150003bd237de3b4115fde88eefa2e188628a791e", "id": "14.Q8b", "kind": "int", "magnitude": 0, "mistakes": {"267291c53bd4ea5e2e6060499d3a41eef54976c30c41ab156608c04155f77910": "une souche a une seule question", "e707efc6853d3ee8c8149b33995dd18b2ae171d4817df104bedbd6aeab38a82b": "combien d'arêtes entre la racine et une feuille d'une souche ?"}, "sign": 1}
📝 Ex 14.Q8b : réponse enregistrée (int).
WB_ANSWER {"hash": "23557a7954e14c28ea8633ff49ee1b23cbbcccb52c63ab19d537db8ac1b8cf94", "id": "14.Q8c", "kind": "bool", "mistakes": {"b61351

**14.Q9 — Ce que repondère AdaBoost**

In [11]:
wb.record("14.Q9a", "B", mistakes={"les mal classés sont multipliés par e^α avec α > 0 : plus grand ou plus petit que 1 ?": "A",
          "relis la mise à jour des poids : les mal classés sont-ils modifiés ?": "C"})
wb.record("14.Q9b", math.log(0.5 / 0.5) + math.log(2 - 1), decimals=3, mistakes={"que vaut ln 1 ?": 1, "à deux classes, que vaut ln(K − 1) ?": 0.693})
wb.record("14.Q9c", "B", mistakes={"regarde comment varie (1 − ε)/ε quand ε diminue": "A"})
wb.record("14.Q9d", False, mistakes={"de quoi le membre m a-t-il besoin avant d'être entraîné ?": True})
wb.record("14.Q9e", 1 - 1 / 4, decimals=2, mistakes={"1/K est la probabilité d'avoir raison au hasard ; quelle erreur fait le hasard ?": 0.25, "avec K = 4 classes, le hasard ne se trompe pas dans 50 % des cas": 0.5,
          "as-tu pris le bon nombre de classes ?": 0.67})

WB_ANSWER {"hash": "d9cab9d5c5631dbe44d6f4a0557f4eac71345d0373fe5f22ff235a37cdf42f7d", "id": "14.Q9a", "kind": "str", "mistakes": {"416eef0606ab25bdb03a2989c588297a689f280d96b188a3cc4df08dc7b7931b": "les mal classés sont multipliés par e^α avec α > 0 : plus grand ou plus petit que 1 ?", "5177e1deb82d05d42ffd7d3dae3b425904c33e3305c159e47cc8625bd60e092d": "relis la mise à jour des poids : les mal classés sont-ils modifiés ?"}}
📝 Ex 14.Q9a : réponse enregistrée (str).
WB_ANSWER {"decimals": 3, "hash": "c5b94c53458c9059de10db7036417a3409f00bcced2326fc9a3babec860d3314", "hash_coarse": "98d35c210f881302bd2e6e3bfed8d72200cbe0e2c3fd8572d7f9d9dea4888052", "id": "14.Q9b", "kind": "float", "magnitude": null, "mistakes": {"35209631d76c15282bc53fc6a10b36de8d180c6e3d70b84a6e2855aa13e5e0e5": "que vaut ln 1 ?", "a1b5daacb4d15aa9c08a2f4fbcb5a86e1b4d8d85657bc777d86a382e3bec5ec9": "à deux classes, que vaut ln(K − 1) ?"}, "sign": 0}
📝 Ex 14.Q9b : réponse enregistrée (float, 3 décimale(s)).
WB_ANSWER {"has

**14.Q10 — Bagging ou boosting ?**

In [12]:
wb.record("14.Q10a", "BOOBOB", mistakes={"relis la ligne « ce qui baisse » du tableau Bilan pour les phrases 3 et 6": "BOBBOO",
          "relis la ligne « bruit des labels » du tableau Bilan pour la phrase 5": "BOOBBB",
          "pour chaque phrase, demande-toi si les membres sont entraînés indépendamment ou l'un après l'autre": "OBBOBO"})
wb.record("14.Q10b", "B", mistakes={"une souche qui underfit souffre-t-elle de biais ou de variance, et que réduit le bagging ?": "A"})
wb.record("14.Q10c", False, mistakes={"relis le §14.4 : que se passe-t-il quand on ajoute des arbres à une forêt ?": True})

WB_ANSWER {"hash": "da2dee8c0c05de2b1973e3cc944932a3ee9cc3cfc69ef23456dc6be58b91abd0", "id": "14.Q10a", "kind": "str", "mistakes": {"2d9ad7c433aeaffb0c1f475edfea81c3d2bb5aea8d157f7622075ed3cdf21a9e": "relis la ligne « ce qui baisse » du tableau Bilan pour les phrases 3 et 6", "495fdcc60ea92311d70e8cce4c5ccb1500a6f7f058c8389b926e3beccd496e44": "pour chaque phrase, demande-toi si les membres sont entraînés indépendamment ou l'un après l'autre", "c9dc78cc1434f7420f0dce7cbe13719d21ec9ec4743316d78bef790ab1605d3b": "relis la ligne « bruit des labels » du tableau Bilan pour la phrase 5"}}
📝 Ex 14.Q10a : réponse enregistrée (str).
WB_ANSWER {"hash": "83177345e172135510021a4f32dfa84d260ee7e07681c133c0e53b1bac879b99", "id": "14.Q10b", "kind": "str", "mistakes": {"7a800039507e3b4d6534ab2a0f19bbf8354c3ef553ae537efcdcbb3295cabb77": "une souche qui underfit souffre-t-elle de biais ou de variance, et que réduit le bagging ?"}}
📝 Ex 14.Q10b : réponse enregistrée (str).
WB_ANSWER {"hash": "1dae8b17d0bb

**14.Q11 — Gradient boosting : corriger les résidus**

In [13]:
wb.record("14.Q11a", "B", mistakes={"0 est-il la meilleure constante pour la MSE ?": "A",
          "la médiane minimise la somme des valeurs absolues ; et la MSE ?": "C",
          "la cible d'un seul exemple résume-t-elle les autres ?": "D"})
wb.record("14.Q11b", 7 - 5, mistakes={"le résidu est la cible moins la prédiction, pas l'inverse": -2})
wb.record("14.Q11c", 5 + 0.1 * 2, decimals=1, mistakes={"multiplie la correction par le learning rate": 7, "on part de la prédiction actuelle, pas de 0": 0.2,
          "on ajoute 0,1 × le résidu, on ne le soustrait pas": 4.8})
wb.record("14.Q11d", True, mistakes={"si chaque arbre ne corrige qu'un dixième de l'erreur, en faut-il plus ou moins ?": False})
wb.record("14.Q11e", True, mistakes={"relis la fiche : que fait un gradient boosting auquel on ajoute des arbres sans contrôle ?": False})

WB_ANSWER {"hash": "430a6164e108c0a6935ade7731d60c61d7379534e494fd90f61568b3569b7bab", "id": "14.Q11a", "kind": "str", "mistakes": {"4bce06f8378cad22354442271363c8f79815b9b757e6015f68873a54ae263247": "la médiane minimise la somme des valeurs absolues ; et la MSE ?", "7c1e04778a365a7472e11833a79b9882f20afee2bb5c8cbabd46c800d8049786": "la cible d'un seul exemple résume-t-elle les autres ?", "dcba9f923aba24b5f5bdd9a6a44a12d9ea36d46d1968036c1e061c3f4e1e7c91": "0 est-il la meilleure constante pour la MSE ?"}}
📝 Ex 14.Q11a : réponse enregistrée (str).
WB_ANSWER {"hash": "0ef82efb2d7f67188dc32b4f8609881b24378b3ec1a7f0a667910da575892d20", "id": "14.Q11b", "kind": "int", "magnitude": 0, "mistakes": {"806fd19bcd06a08ccc1791aa6425e5f8e8fcff020b869801cf52f1cfc532d442": "le résidu est la cible moins la prédiction, pas l'inverse"}, "sign": 1}
📝 Ex 14.Q11b : réponse enregistrée (int).
WB_ANSWER {"decimals": 1, "hash": "cdc4f2319ff5793a5cce07fd5113a760eae4d2529599808fa3826084ee590094", "hash_coarse": 

**14.R1 — L'impureté de Gini d'un nœud**

In [14]:
wb.record("14.R1a", gini([6, 3, 1]), decimals=2, mistakes={"c'est la somme des carrés ; Gini, c'est 1 moins cette somme": 0.46})
wb.record("14.R1b", gini([0, 3, 1]), decimals=3, mistakes={"c'est la somme des carrés ; Gini, c'est 1 moins cette somme": 0.625,
          "les proportions se calculent sur les 4 exemples de l'enfant, pas sur les 10 du parent": 0.9})
wb.record("14.R1c", gini([6, 3, 1]) - 0.4 * gini([0, 3, 1]), decimals=3, mistakes={"pondère l'impureté de l'enfant droit par sa part des exemples (4 sur 10)": 0.165,
          "le gain, c'est l'impureté du parent MOINS la moyenne pondérée des enfants": 0.15,
          "l'enfant droit contient 4 exemples sur 10, pas 6": 0.315})
wb.record("14.R1d", gini([6, 3, 4]), decimals=3, mistakes={"as-tu divisé par la somme des poids (13), et pas par le nombre d'exemples ?": 0.39,
          "as-tu tenu compte du poids 4 de l'exemple de la classe 2 ?": 0.54,
          "c'est la somme des carrés ; Gini, c'est 1 moins cette somme": 0.361})
wb.record("14.R1e", True, mistakes={"les proportions changent-elles si tous les poids sont multipliés par 2 ?": False})

WB_ANSWER {"decimals": 2, "hash": "9031dc17f050a6c9006c91e1b6ced191b975fb8e3a22d6e400b61633d8fa58d1", "hash_coarse": "1626c9f6748af6e1428be79caba298b555c157b5177896b772997082b5c8928f", "id": "14.R1a", "kind": "float", "magnitude": -1, "mistakes": {"f1d669b7e33887c908e383bb58c711557b322ae4b2bf853c74ce423665d41eef": "c'est la somme des carrés ; Gini, c'est 1 moins cette somme"}, "sign": 1}
📝 Ex 14.R1a : réponse enregistrée (float, 2 décimale(s)).
WB_ANSWER {"decimals": 3, "hash": "e6b977a8d9a2b958fb3e96f4e2ef75d51d0a23633a1e8c3a241920468da34b80", "hash_coarse": "2ad8beb9ffe1daf613e493af0550981b26ba1ea6effaff082a53bd564666c380", "id": "14.R1b", "kind": "float", "magnitude": -1, "mistakes": {"dfe9016b88cc3d0ab039bb0647652d8f85c8772d7551440968cc398c6feb3ef2": "les proportions se calculent sur les 4 exemples de l'enfant, pas sur les 10 du parent", "e2b0f173455fa5bb2bf4f5558242cbccbacf9fa9a263a7675710d62c2faa861a": "c'est la somme des carrés ; Gini, c'est 1 moins cette somme"}, "sign": 1}
📝 E

**14.R2 — Induction et généralisation**

In [15]:
wb.record("14.R2a", "B", mistakes={"la conclusion d'un modèle sur un nouveau cas est-elle certaine ?": "A",
          "relis les deux façons de raisonner du ch. 11 : laquelle tire une règle d'observations ?": "C"})
wb.record("14.R2b", False, mistakes={"un bootstrap peut-il contenir ce que l'échantillon ne contient pas ?": True})
wb.record("14.R2c", "B", mistakes={"personne n'invoque ici un expert": "A",
          "l'équipe refuse-t-elle une conclusion qu'imposent les données ?": "C",
          "l'équipe écarte-t-elle des résultats gênants ?": "D"})
wb.record("14.R2d", False, mistakes={"un ensemble est-il entraîné sur autre chose que des données passées ?": True})

WB_ANSWER {"hash": "be2c4f2cd5fbeee58350a4a38affe2becb226a6c18a92459c58d4178cfdcb8cc", "id": "14.R2a", "kind": "str", "mistakes": {"2c7c4fe2fef093838b45091b71ebce10a9512f7c467a255c967c3abeb33cff2a": "la conclusion d'un modèle sur un nouveau cas est-elle certaine ?", "5bfc6f131066bc4f3c090403451c7c41186f519a840998eb7364b7370bfc0499": "relis les deux façons de raisonner du ch. 11 : laquelle tire une règle d'observations ?"}}
📝 Ex 14.R2a : réponse enregistrée (str).
WB_ANSWER {"hash": "9a0cba105bd6eb7b0920e6d8694367cd6863f501db1a53aba07682b46258a7f8", "id": "14.R2b", "kind": "bool", "mistakes": {"8b641b080a8d63489060ac983fc2364c1ae369c97711419e1473afc32e374d7d": "un bootstrap peut-il contenir ce que l'échantillon ne contient pas ?"}}
📝 Ex 14.R2b : réponse enregistrée (bool).
WB_ANSWER {"hash": "3c8c0a56c7333f86217cf9e6dbe34641e15b8d7c587bfd78ac49291df1f15bfa", "id": "14.R2c", "kind": "str", "mistakes": {"79f10ec44c38a819cd3749f17b2c6fb929d4938c4634b0ca6cbb652cd80805ae": "l'équipe refuse-t

**14.R3 — Un-contre-tous et un-contre-un**

In [16]:
wb.record("14.R3a", 5, mistakes={"un-contre-tous oppose chaque classe au reste : combien de classifieurs pour 5 classes ?": 10, "chaque classe a-t-elle son classifieur ?": 4})
wb.record("14.R3b", math.comb(5, 2), mistakes={"un-contre-un : un classifieur par paire de classes, sans compter deux fois la même paire": 20,
          "un-contre-un : un classifieur par paire de classes": 5, "compte les paires de classes distinctes, pas les couples ordonnés ni les classes seules": 25})
wb.record("14.R3c", math.comb(10, 2), mistakes={"un-contre-un : un classifieur par paire de classes, sans compter deux fois la même paire": 90,
          "un-contre-un : un classifieur par paire de classes": 10})
wb.record("14.R3d", True, mistakes={"le classifieur « classe 1 contre classe 3 » a-t-il besoin des exemples des autres classes ?": False})
wb.record("14.R3e", 1 / 4, decimals=2, mistakes={"les positifs sont les exemples d'une seule classe : quelle part du total ?": 0.75, "avec 4 classes, les positifs ne sont pas la moitié des exemples": 0.5})

WB_ANSWER {"hash": "47aee60eabe0805cf7dd553e5ba075b4b8686a694b86a1ed278855c9d7b903e5", "id": "14.R3a", "kind": "int", "magnitude": 0, "mistakes": {"384547dc864171f84f4bd93b8d6c7a225c59f9f37ce644389882fd15c70a11f1": "chaque classe a-t-elle son classifieur ?", "a97ff0e75d6ed6e8f21622b5a9bc5fd9a2b26555eff930594079722a18c89648": "un-contre-tous oppose chaque classe au reste : combien de classifieurs pour 5 classes ?"}, "sign": 1}
📝 Ex 14.R3a : réponse enregistrée (int).
WB_ANSWER {"hash": "92c085a1818a1b884df3d7756c44a39014708a7bfb3836a65380d42a02e905e2", "id": "14.R3b", "kind": "int", "magnitude": 1, "mistakes": {"0e5fcc860448e6e6d21a921f517d5d0ef24826828b66f212ad572b57dd137d5c": "compte les paires de classes distinctes, pas les couples ordonnés ni les classes seules", "9d5a275ec2741f27156c07c1c3543b82b0db502ab5f08b467e5d41bc2ffa6ca2": "un-contre-un : un classifieur par paire de classes, sans compter deux fois la même paire", "d7aad2796d7c168bd287d9b218175baf4f066348629ce87c908e1bd381e708

**Ex 14.1 — Dépouiller un vote pondéré**

In [17]:
wb.record("14.1a", float(scores_141(W_141)[0, 1]), decimals=1, mistakes={"additionne les poids de tous les votants qui prédisent la classe 1 pour l'exemple 0": 1.5,
          "c'est le nombre de voix ; on demande la somme de leurs poids": 2})
wb.record("14.1b", winners_141(W_141)[0], mistakes={"le votant 3 a un poids négatif : as-tu retiré son poids à la classe 0 ?": 0,
          "recalcule le score de la classe 2 : un seul votant la prédit": 2})
wb.record("14.1c", winners_141(W_141)[1], mistakes={"le poids du votant 3 est négatif, et une égalité va à la plus petite classe : recalcule les deux scores": 1,
          "recalcule le score de la classe 2 pour l'exemple 1": 2})
wb.record("14.1d", winners_141(W_141), mistakes={"c'est le vote sans les poids ; ajoute le poids de chaque votant": [0, 1, 0, 0],
          "le votant 3 a un poids négatif : as-tu retiré son poids aux classes qu'il prédit ?": [0, 1, 2, 0],
          "une égalité donne la plus petite classe : vérifie l'exemple 1": [1, 1, 2, 2]})
wb.record("14.1e", winners_141(np.ones(5)), mistakes={"sans les poids, chaque votant a une voix ; en cas d'égalité, la plus petite classe gagne": [1, 1, 2, 2],
          "as-tu gardé les poids ? Donne une voix à chaque votant": [1, 0, 2, 2]})
wb.record("14.1f", 0, mistakes={"avec ce poids, les deux classes sont à égalité : qui gagne une égalité ?": -1,
          "on demande la PLUS PETITE valeur entière : essaie une valeur plus petite": 1})
wb.record("14.1g", True, mistakes={"multiplier tous les scores par 3 change-t-il leur ordre ?": False})
wb.record("14.1h", False, mistakes={"ajouter 10 à chaque voix ajoute à chaque classe 10 fois son NOMBRE de voix : refais l'exemple 1": True})

WB_ANSWER {"alt_coarse": ["09f31c87eab08b3e6e5b49ae9eb8334282b8965deb5efbbcd49a33c2f7dca596"], "decimals": 1, "hash": "c0685537707e09b9447e3d134c19fa99eb40dd06330d9b7838c4829801f7f356", "hash_coarse": "108f17e557af8a90507389863cc6989e71af59651cc0c60633cfd2493c14cf72", "id": "14.1a", "kind": "float", "magnitude": 0, "mistakes": {"88fa0014831fc2bbc0126d0b9ca9ed93043ca471fc3fa56c5ffd9a6b41e94f23": "additionne les poids de tous les votants qui prédisent la classe 1 pour l'exemple 0", "e934112effe6299c2d56547dff9d4325de297d09f594b0a7fd5553a2348ff8c0": "c'est le nombre de voix ; on demande la somme de leurs poids"}, "sign": 1}
📝 Ex 14.1a : réponse enregistrée (float, 1 décimale(s)).
WB_ANSWER {"hash": "bc12c3d1cedce398bde534506499a8aec7b4046ad18d4dcbf1375d86c5424efd", "id": "14.1b", "kind": "int", "magnitude": 0, "mistakes": {"5dc6b1a4dd890a003d1641342602a40d51dc27ac2f8e00bfea234bdd4ac1f03d": "recalcule le score de la classe 2 : un seul votant la prédit", "7331d043e0122ebf3fd5aeae77020e1f77a

**Ex 14.2 — Additionner les voix de trois droites, région par région**

In [18]:
wb.record("14.2a", score_142(*REGIONS_142['C']), mistakes={"la région « C » ne reçoit que le poids de C, signe compris": 3, "la région « C » est du côté positif de C : elle reçoit son poids": 0})
wb.record("14.2b", score_142(1.5, 1.2), mistakes={"vérifie les trois conditions pour ce point, dont x₁ + x₂ ≤ 3": 4, "le point est-il du côté positif de A et de B ?": -3})
wb.record("14.2c", int(score_142(0.5, 0.5) > 0), mistakes={"le point est classé 1 seulement si son score est strictement positif : calcule son score": 1,
          "réponds par la classe, 0 ou 1": -3})
wb.record("14.2d", sum(score_142(*p) > 0 for p in REGIONS_142.values()), mistakes={"recalcule le score de chaque région, et compte celles de score strictement positif": 3,
          "recalcule le score de chaque région, signe compris": 5})
wb.record("14.2e", [int(score_142(*p) > 0) for p in POINTS_142], mistakes={"pour P₂, calcule x₁ + x₂ avant de conclure sur C": [1, 0, 0, 0, 1]})
wb.record("14.2f", sum(score_142(*p, wC=-1) > 0 for p in REGIONS_142.values()), mistakes={"recalcule les sept scores avec w_C = −1": 4, "recalcule les sept scores avec w_C = −1, signe compris": 7})
wb.record("14.2g", float(np.mean([int(p[0] >= 1) == t for p, t in zip(POINTS_142, TRUE_142)])), decimals=1, mistakes={"A prédit 1 seulement de son côté positif (x₁ ≥ 1) : compare point par point": 0.4,
          "compare les prédictions de A seul aux vraies classes, point par point": 1})
wb.record("14.2h", (2 + 2 - 3) / 2, decimals=1, mistakes={"remplace chaque indicatrice par (s + 1)/2 : as-tu gardé le facteur 1/2 ?": 1,
          "la constante dépend du signe de chaque poids : as-tu pris w_C avec son signe ?": 3.5})

WB_ANSWER {"hash": "c32abb61cd46d2d13d932bd2d18ecc4b3f754fc2f10b97f778689802b808c1af", "id": "14.2a", "kind": "int", "magnitude": 0, "mistakes": {"4489652ce365b56302d5eba769d88cc5672d66115afec6938f6ac0e669a3b5fd": "la région « C » est du côté positif de C : elle reçoit son poids", "f553561a99e797963286e55aca696306c20b7fb6e4977ca6b47fdb126498a863": "la région « C » ne reçoit que le poids de C, signe compris"}, "sign": -1}
📝 Ex 14.2a : réponse enregistrée (int).
WB_ANSWER {"hash": "d19f81536f6116df340eeb316d05efc06c75b8e166e7cfb26b84a4406ea7079f", "id": "14.2b", "kind": "int", "magnitude": 0, "mistakes": {"a1580f9ea0a8a029eb8a5d531c591877fe0e47319c22dcb10701f1d3de55a024": "vérifie les trois conditions pour ce point, dont x₁ + x₂ ≤ 3", "d02b4c656dff23dbad23032d704bd3b5b285a397eff684597bea1755e6c9af6c": "le point est-il du côté positif de A et de B ?"}, "sign": 1}
📝 Ex 14.2b : réponse enregistrée (int).
WB_ANSWER {"hash": "02f72501e2739731b8a4f51dbf0b1d04b351dd01f2bcc6600135076f9d62a042", 

**Ex 14.3 — Probabilité d'être oublié par un bootstrap : vers 36,8 %**

In [19]:
wb.record("14.3a", (7 / 8) ** 4, decimals=3, mistakes={"c'est la probabilité d'être tiré à chaque tirage ; on demande de ne jamais l'être": 0.0002,
          "les probabilités des tirages indépendants se multiplient, elles ne s'additionnent pas": 0.5,
          "c'est la probabilité d'être tiré au moins une fois": 0.414,
          "as-tu échangé n et d ? Il y a 4 tirages parmi 8 exemples": 0.1})
wb.record("14.3b", 8 * (1 - (7 / 8) ** 4), decimals=2, mistakes={"c'est le nombre de tirages ; des tirages peuvent tomber sur le même exemple": 4,
          "c'est le nombre moyen d'exemples JAMAIS tirés": 4.69})
wb.record("14.3c", 0.9 ** 10, decimals=4, mistakes={"c'est la probabilité d'être tiré au moins une fois": 0.6513,
          "calcule 0,9 puissance 10 lui-même : n = 10 est loin de la limite": 0.3679})
wb.record("14.3d", 0.999 ** 1000, decimals=4, mistakes={"c'est la probabilité d'être tiré au moins une fois": 0.6323, "calcule 0,999 puissance 1 000 lui-même, sans passer à la limite": 0.3679})
wb.record("14.3e", math.exp(-1), decimals=4, mistakes={"relis ce que demande e) : l'exposant est −1": 2.7183, "relis ce que demande e) : une seule exponentielle, sans complément": 0.6321})
wb.record("14.3f", 100 * math.log(1 - 1 / 100), decimals=4, mistakes={"le logarithme est népérien, pas décimal": -0.4365, "c'est l'approximation −1 ; calcule la valeur exacte": -1})
wb.record("14.3g", 1000 * (1 - 0.999 ** 1000), decimals=1, mistakes={"c'est le nombre moyen d'exemples JAMAIS tirés": 367.7, "c'est l'arrondi avec la limite e⁻¹ ; calcule avec (1 − 1/1 000)^1 000": 632.1})
wb.record("14.3h", 1 - (1 - math.exp(-1)) ** 10, decimals=3, mistakes={"c'est la probabilité d'être out-of-bag pour TOUS les arbres": 0.0,
          "c'est la probabilité d'être dans les dix bootstraps": 0.010,
          "c'est la probabilité pour un seul arbre": 0.368})

WB_ANSWER {"decimals": 3, "hash": "6ae566f3d6259aaebdf40cda7b6662714867e0b9c27b6bf94e789412525d71da", "hash_coarse": "77925b05f12862d658136de926186f79f90732416f78b392148392b6eca514ba", "id": "14.3a", "kind": "float", "magnitude": -1, "mistakes": {"6a721c2188cdd14da5c82ac77a3325340c83a2badb0ee78153c5712ad6e6740d": "les probabilités des tirages indépendants se multiplient, elles ne s'additionnent pas", "86cc30a2b94d99b02eaefd67cf27591357ca75aeed233f353fa1ad124355fb89": "c'est la probabilité d'être tiré à chaque tirage ; on demande de ne jamais l'être", "96b7a9d61a0698dd4a86336bc9546e7fa1b7225c2dc231bf3289049b8fec1831": "c'est la probabilité d'être tiré au moins une fois", "b4a25750061f411acf7df390c65f0b2b0a47e667767536d8c4aa2af851d9a71c": "as-tu échangé n et d ? Il y a 4 tirages parmi 8 exemples"}, "sign": 1}
📝 Ex 14.3a : réponse enregistrée (float, 3 décimale(s)).
WB_ANSWER {"decimals": 2, "hash": "b644bde8809dce3169692e5f7e2c8b2cc5da5624e35bb41fb804b95ed825332f", "hash_coarse": "5e7237

**Ex 14.5 — Quand la majorité de votants indépendants se trompe**

In [20]:
wb.record("14.5a", majority_right(3, 0.7), decimals=3, mistakes={"il y a 3 façons d'avoir exactement deux votants justes : as-tu le coefficient binomial ?": 0.49,
          "c'est la probabilité que les trois aient raison ; ajoute le cas de deux justes": 0.343,
          "la majorité est juste aussi quand les trois ont raison": 0.441})
wb.record("14.5b", majority_right(5, 0.7), decimals=3, mistakes={"la majorité de 5, c'est 3, 4 ou 5 votants justes : as-tu tous les termes ?": 0.309,
          "as-tu ajouté le terme des cinq votants justes ?": 0.669})
wb.record("14.5c", majority_right(3, 0.4), decimals=3, mistakes={"il y a 3 façons d'avoir exactement deux votants justes : as-tu le coefficient binomial ?": 0.16,
          "c'est la probabilité que la majorité se trompe": 0.648})
wb.record("14.5d", majority_right(11, 0.6), decimals=4, mistakes={"la majorité de 11 votants, c'est au moins 6 votants justes, pas 5": 0.9006,
          "la majorité de 11 votants, c'est au moins 6 votants justes, pas 7": 0.5328,
          "c'est la probabilité que la majorité se trompe": 0.2465})
wb.record("14.5e", math.comb(5, 3) * 0.7 ** 3 * 0.3 ** 2, decimals=4, mistakes={"il y a 10 façons de choisir les trois votants justes parmi cinq": 0.0309,
          "on demande exactement trois votants justes : un seul terme de la somme": 0.8369})
wb.record("14.5f", 0.7, decimals=1, mistakes={"les trois votes sont toujours identiques : la loi binomiale s'applique-t-elle ?": 0.8})
wb.record("14.5g", next(M for M in range(1, 50, 2) if majority_right(M, 0.7) >= 0.9), mistakes={"calcule la probabilité pour ce nombre de votants : atteint-elle 0,9 ?": 7, "existe-t-il un nombre impair plus petit qui atteint 0,9 ?": 11})
wb.record("14.5h", majority_right(101, 0.51), decimals=3, mistakes={"c'est la probabilité que la majorité se trompe": 0.42, "c'est p ; on demande la probabilité que la majorité ait raison": 0.51})
wb.record("14.5i", majority_right(25, 0.45), decimals=3, mistakes={"c'est la probabilité que la majorité se trompe": 0.694, "c'est p ; on demande la probabilité que la majorité ait raison": 0.45})

WB_ANSWER {"decimals": 3, "hash": "233a74ffbbc067cab0524d2ad40b61e422b998fee85b1a44627748d07b1bfa06", "hash_coarse": "52dac454b3af8fad42d622b43b947d23061186eeb9acbb64b95a1d5f2dbc846d", "id": "14.5a", "kind": "float", "magnitude": -1, "mistakes": {"1fb2ca763590e8e9ad8159a83dd3634b7c63df98b10aeff5d5806945511d55aa": "il y a 3 façons d'avoir exactement deux votants justes : as-tu le coefficient binomial ?", "866e982bef534d4448e9405fbd86618ce49e416445d8be41b3691d6551f5bae6": "c'est la probabilité que les trois aient raison ; ajoute le cas de deux justes", "e5db71ffb564454d92fe4ec3c7f4e768a73bcd77d2347a310b8f188004e68082": "la majorité est juste aussi quand les trois ont raison"}, "sign": 1}
📝 Ex 14.5a : réponse enregistrée (float, 3 décimale(s)).
WB_ANSWER {"decimals": 3, "hash": "c3ff7972cb339606958014740980abd3d516abac739b313640528d308de88c01", "hash_coarse": "0c43d2a6304d85862608cbd2f41247429fb077a30d3935f3a90b2fd22675d8fb", "id": "14.5b", "kind": "float", "magnitude": -1, "mistakes": {"

**Ex 14.6 — Une itération d'AdaBoost à la main**

In [21]:
wb.record("14.6a", R_146[0]['err'], decimals=3, mistakes={"cherche une souche qui ne laisse qu'un seul exemple du mauvais côté": 0.25,
          "c'est l'accuracy ; on demande l'erreur pondérée": 0.875})
wb.record("14.6b", R_146[0]['alpha'], decimals=3, mistakes={"le logarithme est népérien, pas décimal": 0.845, "c'est (1 − ε)/ε ; prends son logarithme": 7,
          "la formule de la fiche n'a pas de facteur 1/2": 0.973,
          "à deux classes, ln(K − 1) vaut 0": 2.639})
wb.record("14.6c", float(W_AFTER_1[1]), decimals=3, mistakes={"as-tu normalisé les poids pour qu'ils fassent 1 au total ?": 0.875,
          "le poids de l'exemple mal classé est multiplié par e^α, puis on normalise": 0.125})
wb.record("14.6d", float(W_AFTER_1[0]), decimals=4, mistakes={"les poids des exemples bien classés ne changent qu'à la normalisation : divise par le nouveau total": 0.125})
wb.record("14.6e", [R2_146[1]['s'], R2_146[1]['g']], decimals=1, mistakes={"avec les nouveaux poids, la souche du tour 1 a une erreur bien plus grande : recalcule": [4.5, 0],
          "le côté gauche de ta souche prédit l'autre classe : vérifie-la sur x = 1": [1.5, 1]})
wb.record("14.6f", R2_146[1]['err'], decimals=4, mistakes={"additionne les poids des exemples mal classés par la souche du tour 2, avec les poids de c) et d)": 0.25,
          "c'est l'erreur de la souche du tour 1 avec les nouveaux poids": 0.5})
wb.record("14.6g", R2_146[1]['alpha'], decimals=3, mistakes={"le logarithme est népérien, pas décimal": 0.778, "c'est (1 − ε)/ε ; prends son logarithme": 6,
          "la formule de la fiche n'a pas de facteur 1/2": 0.896})
wb.record("14.6h", vote_146(R2_146), mistakes={"là où les deux souches se contredisent, compare leurs poids α": [0, 1, 1, 1, 1, 1, 1, 1]})
wb.record("14.6i", R3_146[2]['err'], decimals=4, mistakes={"as-tu mis à jour les poids après le tour 2 (erreurs de la souche 2 multipliées par e^α₂, puis normalisation) ?": 0.3571,
          "les poids ne sont plus uniformes au tour 3 : utilise les poids mis à jour": 0.625,
          "as-tu bien compté tous les exemples que cette souche classe mal (à gauche et à droite) ?": 0.0417})
wb.record("14.6j", R3_146[2]['alpha'], decimals=3, mistakes={"le logarithme est népérien, pas décimal": 0.580,
          "l'erreur de i) se calcule avec les poids mis à jour après le tour 2 : refais i) d'abord": 0.588,
          "la formule de la fiche n'a pas de facteur 1/2": 0.668, "la formule de la fiche n'a pas de facteur 1/2 (et garde α non arrondi)": 0.667})
wb.record("14.6k", vote_146(R3_146), mistakes={"additionne les α des souches qui votent pour chaque classe, exemple par exemple": [0, 0, 0, 0, 1, 1, 1, 1],
          "pour x = 3 et x = 4, compare la somme des α de chaque classe": [0, 1, 1, 1, 1, 1, 1, 1]})
wb.record("14.6l", math.log(0.7 / 0.3) + math.log(3 - 1), decimals=3, mistakes={"avec K = 3, ajoute ln(K − 1)": 0.847, "le terme ajouté est ln(K − 1), pas ln K": 1.946,
          "le logarithme est népérien, pas décimal": 0.669})
wb.record("14.6m", 0.55 < 1 - 1 / 3, mistakes={"avec K = 3 classes, le seuil de rejet n'est pas 0,5 : compare 0,55 à 1 − 1/K": False})

WB_ANSWER {"alt_coarse": ["e89409b02e821a52579148623773658f7b9be338d456f43a05af241d72acbddf"], "decimals": 3, "hash": "1ff80b49e6fae32e7abc24545a645af190c269d1a3dd10666e1fb3faf6f51a8d", "hash_coarse": "6b3fbeb171db3fda9c6289d0aa713c286e44fbfc74d99eab1498d69590fa9c17", "id": "14.6a", "kind": "float", "magnitude": -1, "mistakes": {"b598a865bf9d0060974cadc3df2050797485c4b89ec31c2cb10ca04b7fcb9d6d": "c'est l'accuracy ; on demande l'erreur pondérée", "f89de8fad02fbe03d0d49a61c7a76c4e2f68aca73bd39d3cbbd232914f3f4f16": "cherche une souche qui ne laisse qu'un seul exemple du mauvais côté"}, "sign": 1}
📝 Ex 14.6a : réponse enregistrée (float, 3 décimale(s)).
WB_ANSWER {"decimals": 3, "hash": "642db69619caeeb62f2096f0a896c2784d86aaffc1bea8dde303b5d90e9cee7f", "hash_coarse": "e15731e50f4fdcd15abe89326a73f5c5f0cf36f62b330a66254d8284c8d065ab", "id": "14.6b", "kind": "float", "magnitude": 0, "mistakes": {"062232996fe14fdbe2bd4ab407e9abf8e927ec0be8d339471e73a7d779826907": "la formule de la fiche n'a 

**Ex 14.7 — Deux itérations de gradient boosting sur cinq points**

In [22]:
wb.record("14.7a", int(Y_147.mean()), mistakes={"c'est la médiane ; la MSE demande une autre constante": 6})
wb.record("14.7b", [int(v) for v in Y_147 - Y_147.mean()], mistakes={"le résidu est la cible moins la prédiction": [4, 3, -1, -2, -4]})
wb.record("14.7c", T1_147[0][0], decimals=1, mistakes={"compare les sommes des carrés des écarts des quatre seuils, sur les résidus": 3.5})
wb.record("14.7d", [T1_147[0][1], T1_147[0][2]], decimals=3, mistakes={"la souche apprend les résidus : chaque feuille prédit la moyenne des résidus, pas des cibles y": [1.5, 7.333],
          "as-tu pris le seuil de la question c) ?": [-2.0, 3.0]})
wb.record("14.7e", float(F1_147[0]), decimals=2, mistakes={"multiplie la valeur de la feuille par le learning rate 0,5": 1.5,
          "x = 1 est dans la feuille gauche, x ≤ 2,5": 6.17,
          "F₁ = F₀ + 0,5 × la feuille : as-tu ajouté F₀ ?": -1.75})
wb.record("14.7f", float(np.mean((Y_147 - F1_147) ** 2)), decimals=3, mistakes={"divise la somme des carrés par 5, le nombre d'exemples": 15.375,
          "c'est la MSE avant la première étape": 9.2,
          "as-tu multiplié la feuille par le learning rate 0,5 dans F₁ ?": 1.033})
wb.record("14.7g", T2_147[1][0], decimals=1, mistakes={"la deuxième souche s'ajuste aux NOUVEAUX résidus y − F₁ : recompare les quatre seuils": 2.5,
          "as-tu calculé F₁ avec le learning rate 0,5 ?": 4.5})
wb.record("14.7h", predict_147(3, 0.5, T2_147), decimals=4, mistakes={"as-tu ajouté la contribution de la deuxième souche ?": 6.1667,
          "multiplie aussi la feuille de la deuxième souche par 0,5": 4.9444})
wb.record("14.7i", predict_147(4.2, 0.5, T2_147), decimals=4, mistakes={"as-tu ajouté la contribution de la deuxième souche ?": 6.1667,
          "multiplie aussi la feuille de la deuxième souche par 0,5": 8.0})
wb.record("14.7j", float(np.mean((Y_147 - F2_147) ** 2)), decimals=4, mistakes={"as-tu ajouté la deuxième souche avant de calculer la MSE ?": 3.075, "divise la somme des carrés par 5": 6.9722})
wb.record("14.7k", predict_147(1, 1.0, boost_147(1.0, 1)[1]), decimals=1, mistakes={"avec η = 1, la feuille s'ajoute entière": 3.25, "F₁ = F₀ + la feuille : as-tu ajouté F₀ ?": -3.5})

WB_ANSWER {"hash": "669dc099b3ef8755893ae9e768f2ff0ae17e88840d2f2a437dfa4b360c0f08be", "id": "14.7a", "kind": "int", "magnitude": 0, "mistakes": {"2d1e0d16420d66cc4fcd4cf74d5f5d47e8fad835e3d7f8734c92000d731ae6b3": "c'est la médiane ; la MSE demande une autre constante"}, "sign": 1}
📝 Ex 14.7a : réponse enregistrée (int).
WB_ANSWER {"decimals": 0, "element_hashes": ["76d4fe7f699024d94d6089f1cb0c2d749fb929bf6eeacaedbc1b12ee7ddafbe8", "c876d3b208d758c85825983013dc993c82f14b5f5405e1c76dc514e91a0b4125", "dc85d83f2572ab8ad9899a4ae6c1d40c19589c88ad2b37eca034ed34230d07bb", "6ff50fb5ea4c103f4ed3304fe130728bf527d476fd960f00b5417a5fa03353b6", "8ee1dce96ebf476f5b36b0bbc11b5a00c11fb46a8ec5e8a4b590817f30ca0f7c"], "hash": "83c25e5b02d32ea8fdab6519ea2a6942dcf26c71fe3ea64a1abaedb0672cdc49", "id": "14.7b", "integer": true, "kind": "array", "mistakes": {"d2c5af598ed604c2566f138407ba790e9de54e6941566b218e4021122fde56d3": "le résidu est la cible moins la prédiction"}, "shape": [5]}
📝 Ex 14.7b : réponse enr

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu dire quand un ensemble bat ses membres, et pourquoi il ne corrige pas un biais commun ?
2. Sais-tu dérouler à la main un vote pondéré, un tour d'AdaBoost et une étape de gradient boosting ?
3. Sais-tu choisir entre une forêt et un gradient boosting, et régler le nombre d'arbres et le learning rate ?

**Pour aller plus loin** : les articles de Breiman (1996, 2001), de Freund et Schapire (1997) et de Friedman (2001), et le guide utilisateur « Ensembles » de scikit-learn, cités dans la fiche. La suite : le ch. 15 (scikit-learn), qui mettra ces ensembles dans des pipelines et des recherches d'hyperparamètres.

*Les exercices de code 14.11 à 14.28 (parties A à D, ta librairie `ensemble`) seront ajoutés à la prochaine session de génération ; relance alors `python tools/start_chapter.py 14` pour obtenir le notebook complet.*